# 👑🎨 KING2-IMAGE — تدريب مولّد صور (SDXL + LoRA) على Google Colab

تدريب نموذج صور خاص بعائلة KING2 فوق **Stable Diffusion XL** باستخدام LoRA
على عيّنة من داتا `jackyhate/text-to-image-2M`.

**قبل التشغيل:**
1. القائمة العلوية: **Runtime → Change runtime type → T4 GPU** ثم Save
2. أيقونة المفتاح 🔑 يسار الشاشة (Secrets) → أضف سرًا اسمه `HF_TOKEN` = توكن HuggingFace (بصلاحية Write) → فعّل Notebook access
3. **Runtime → Run all**

> ⚠️ Colab المجاني قد يفصل في الجلسات الطويلة — خلِّ التبويب مفتوحًا. لو فصل، أعد التشغيل (الـ checkpoints محفوظة كل 500 خطوة).

In [ ]:
# 1. تثبيت المكتبات + سكربت التدريب الرسمي
# diffusers من المصدر لأن سكربت التدريب (main) يحتاج نسخة dev
!pip install -q git+https://github.com/huggingface/diffusers
!pip install -qU accelerate transformers peft bitsandbytes datasets huggingface_hub webdataset Pillow
!pip uninstall -y wandb -q   # wandb في Colab معطوب ويوقف السكربت — وما نحتاجه (report_to=none)
!wget -q https://raw.githubusercontent.com/huggingface/diffusers/main/examples/text_to_image/train_text_to_image_lora_sdxl.py -O train_text_to_image_lora_sdxl.py
!sed -i 's/is_wandb_available()/False/g' train_text_to_image_lora_sdxl.py   # تعطيل wandb داخل السكربت نهائياً
print('✅ تم التثبيت')

In [ ]:
# 2. تسجيل الدخول HuggingFace (من سر Colab: HF_TOKEN)
from huggingface_hub import login, whoami
from google.colab import userdata
login(userdata.get('HF_TOKEN'))
print('Logged in as:', whoami()['name'])

In [ ]:
# 3. بث عيّنة من الداتا وحفظها كـ imagefolder
import os, json
from datasets import load_dataset

N_SAMPLES = 3000   # عيّنة مناسبة للطبقة المجانية
RES = 1024          # خفّضها لـ 768 لو حصل OOM
DATA_DIR = '/content/king2_img_data'
os.makedirs(DATA_DIR, exist_ok=True)

ds = load_dataset('jackyhate/text-to-image-2M', split='train', streaming=True)
meta, kept = [], 0
for ex in ds:
    if kept >= N_SAMPLES:
        break
    try:
        img = ex['jpg'].convert('RGB')
        prompt = (ex.get('json') or {}).get('prompt', '').strip()
        if not prompt:
            continue
        w, h = img.size; s = min(w, h)
        img = img.crop(((w-s)//2,(h-s)//2,(w-s)//2+s,(h-s)//2+s)).resize((RES, RES))
        fn = f'{kept:06d}.jpg'
        img.save(os.path.join(DATA_DIR, fn), quality=92)
        meta.append({'file_name': fn, 'prompt': prompt}); kept += 1
        if kept % 250 == 0:
            print(f'  حفظ {kept}/{N_SAMPLES}')
    except Exception:
        continue

with open(os.path.join(DATA_DIR, 'metadata.jsonl'), 'w', encoding='utf-8') as f:
    for m in meta:
        f.write(json.dumps(m, ensure_ascii=False) + '\n')
print(f'✅ جاهز: {kept} صورة')

In [ ]:
# 4. إعداد accelerate
from accelerate.utils import write_basic_config
write_basic_config(mixed_precision='fp16')
print('✅ accelerate config')

In [ ]:
# 5. التدريب — SDXL LoRA (إعدادات موفّرة للذاكرة 16GB)
!accelerate launch train_text_to_image_lora_sdxl.py \
  --pretrained_model_name_or_path="stabilityai/stable-diffusion-xl-base-1.0" \
  --pretrained_vae_model_name_or_path="madebyollin/sdxl-vae-fp16-fix" \
  --train_data_dir="/content/king2_img_data" \
  --caption_column="prompt" \
  --resolution=1024 --center_crop --random_flip \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --gradient_checkpointing \
  --use_8bit_adam \
  --mixed_precision="fp16" \
  --max_train_steps=1500 \
  --learning_rate=1e-4 \
  --lr_scheduler="cosine" --lr_warmup_steps=0 \
  --rank=16 \
  --checkpointing_steps=500 \
  --seed=42 \
  --output_dir="/content/king2-image-lora" \
  --report_to="none"

In [ ]:
# 6. اختبار النموذج — توليد صورة
import torch
from diffusers import DiffusionPipeline
pipe = DiffusionPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0', torch_dtype=torch.float16
).to('cuda')
pipe.load_lora_weights('/content/king2-image-lora')
image = pipe('a futuristic royal palace at sunset, highly detailed, 8k, cinematic',
             num_inference_steps=30, guidance_scale=7.0).images[0]
image.save('/content/king2_test.png')
image

In [ ]:
# 7. رفع النموذج باسم king2-image
from huggingface_hub import HfApi, create_repo
REPO_ID = 'RASHID778/king2-image'
create_repo(REPO_ID, repo_type='model', exist_ok=True)
HfApi().upload_folder(
    folder_path='/content/king2-image-lora',
    repo_id=REPO_ID, repo_type='model',
    commit_message='KING2-IMAGE: SDXL LoRA trained on text-to-image-2M subset',
)
print(f'✅ تم الرفع: https://huggingface.co/{REPO_ID}')